# 4 — Agent flow

Runnable code from chapter 4 of *AI Agents*, generated from the book's own sources.

Cells follow the order of the chapter, and the headings below carry the book's section and listing numbers, so you can read and run side by side.

Some cells set up state the book does not print — imports, the API client, helpers introduced earlier. They are included so the notebook runs on its own, and are marked *setup*. Run it from top to bottom.

Your results will differ in wording from the printed ones: these are live model calls.

## 4.3 The agent loop

**Listing 4.1** — The agent loop as an abstract execution schema

> The book does not execute this listing. It needs values or a service you must supply yourself, so running it as printed will not work unedited.

In [ ]:
state = initialize_state(user_input)

while not should_stop(state):
    action = select_next_action(state)
    observation = execute_action(action, state)
    state = update_state(state, action, observation)

response = finalize_response(state)

## 4.4 A minimal agent loop in Python

*setup — not printed in the book*

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv()

client = OpenAI()

CHAT_MODEL = os.environ["CHAT_MODEL"]

In [ ]:
import re

RETURN_POLICY = [
    {"id": "§2.1", "text": "Standard items may be returned within "
     "14 days of delivery."},
    {"id": "§4.2", "text": "Refunds are issued to the original "
     "payment method within 5 business days after the returned item "
     "has been received."},
    {"id": "§4.3", "text": "Discounted items are excluded from "
     "refunds but may be exchanged within 14 days."},
]

def tokens(text: str):
    return set(re.findall(r"[a-z0-9]+", text.lower()))

def search_return_policy(query: str):
    words = tokens(query)
    ranked = sorted(
        RETURN_POLICY,
        key=lambda p: len(words & tokens(p["text"])),
        reverse=True,
    )
    return {"passages": ranked[:2]}

ORDERS = {
    "A-1001": {"refund_issued": True, "refund_date": "2026-06-24",
               "amount_eur": 79.9},
    "A-1002": {"refund_issued": False,
               "reason": "returned item not yet received"},
}

def get_refund_status(order_id: str):
    if order_id in ORDERS:
        return ORDERS[order_id]
    return {"error": f"unknown order id '{order_id}'"}

In [ ]:
return_policy_tool = {
  "type": "function",
  "function": {
    "name": "search_return_policy",
    "description": "Search the return policy for relevant passages.",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {"type": "string",
                  "description": "Search terms for the policy"}
      },
      "required": ["query"],
      "additionalProperties": False
    }
  }
}

refund_status_tool = {
  "type": "function",
  "function": {
    "name": "get_refund_status",
    "description": "Look up the refund status of an order.",
    "parameters": {
      "type": "object",
      "properties": {
        "order_id": {"type": "string",
                     "description": "Order id, e.g. 'A-1001'"}
      },
      "required": ["order_id"],
      "additionalProperties": False
    }
  }
}

TOOL_IMPLEMENTATIONS = {
    "search_return_policy": search_return_policy,
    "get_refund_status": get_refund_status,
}

In [ ]:
def run_agent(user_input: str, max_steps: int = 6):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a customer support agent. Ground policy "
                "statements in retrieved policy passages and account "
                "statements in tool results. If required information "
                "is missing, ask the user instead of guessing."
            ),
        },
        {"role": "user", "content": user_input},
    ]
    for step in range(max_steps):
        resp = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=messages,
            tools=[return_policy_tool, refund_status_tool],
            tool_choice="auto",
            temperature=0.2,
            seed=42,
        )
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content  # stopping condition: answer is ready
        messages.append(
            {"role": "assistant", "content": None,
             "tool_calls": msg.tool_calls}
        )
        for tc in msg.tool_calls:  # delegation
            implementation = TOOL_IMPLEMENTATIONS[tc.function.name]
            args = json.loads(tc.function.arguments)
            observation = implementation(**args)
            print(f"Step {step + 1}: {tc.function.name}({args})")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(observation),  # observation
            })
    return ("I could not resolve this request within the step "
            "budget; escalating to a human agent.")

In [ ]:
answer = run_agent(
    "Has the refund for order A-1001 been issued? And how long do I "
    "have to return another item from the same delivery?"
)
print(answer)

## 4.7 State, working memory, and context engineering

**Listing 4.2** — Compaction in one function: older tool observations shrink to one-line summaries that keep the load-bearing content

In [ ]:
def summarize_observation(content: str):
    data = json.loads(content)
    if "passages" in data:
        parts = [p["id"] + ": " + " ".join(p["text"].split()[:8])
                 for p in data["passages"]]
        return "[" + "; ".join(parts) + "]"
    return "[" + ", ".join(
        f"{k}={v}" for k, v in list(data.items())[:3]) + "]"

PREFIX = "Tool result"

def compress_messages(messages, keep_last: int = 2):
    cutoff = len(messages) - keep_last
    out = []
    for i, m in enumerate(messages):
        if i < cutoff and m["content"].startswith(PREFIX):
            tool = m["content"].split(":", 1)[0]
            payload = m["content"].split(":", 1)[1].strip()
            out.append({"role": "user", "content": (
                f"{tool}: " + summarize_observation(payload))})
        else:
            out.append(m)
    return out

policy_obs = search_return_policy("return window refund")
status_obs = get_refund_status("A-1001")

state = [
    {"role": "system", "content": (
        "You are a customer support agent. Ground policy statements "
        "in retrieved policy passages and account statements in "
        "tool results. Cite policy passages by their § id.")},
    {"role": "user", "content": (
        "Has the refund for order A-1001 been issued? And how long "
        "do I have to return another item from the same delivery?")},
    {"role": "user", "content": (
        f"{PREFIX} (search_return_policy): "
        + json.dumps(policy_obs))},
    {"role": "user", "content": (
        f"{PREFIX} (get_refund_status): "
        + json.dumps(status_obs))},
]

def answer_over(messages):
    resp = client.chat.completions.create(
        model=CHAT_MODEL, messages=messages,
        temperature=0.2, seed=42)
    return (resp.choices[0].message.content,
            resp.usage.prompt_tokens)

full_answer, full_tokens = answer_over(state)
compressed = compress_messages(state, keep_last=1)
comp_answer, comp_tokens = answer_over(compressed)

print(f"prompt tokens - full: {full_tokens}, "
      f"compressed: {comp_tokens}")
print("\nCompressed-state answer:\n", comp_answer)

## 4.9.1 Reactive: the baseline, measured

**Listing 4.3** — A measurement shim: run any variant and count its total tokens

In [ ]:
def count_tokens(run_fn, *args, **kwargs):
    total = {"tokens": 0}
    original = client.chat.completions.create

    def counted(**kw):
        resp = original(**kw)
        total["tokens"] += resp.usage.total_tokens
        return resp

    client.chat.completions.create = counted
    try:
        result = run_fn(*args, **kwargs)
    finally:
        client.chat.completions.create = original
    return result, total["tokens"]

## 4.9.2 Plan-and-execute: the decomposition becomes inspectable

**Listing 4.4** — Plan-and-execute: the decomposition becomes an inspectable JSON artifact before any tool runs

In [ ]:
PLANNER_PROMPT = (
    "You are the planner for a customer support agent. Decompose "
    "the user's request into the smallest sufficient sequence of "
    "steps. Return JSON only, in the form {\"steps\": [{\"action\": "
    "..., \"input\": {...}, \"why\": ...}]}. Allowed actions: "
    "'search_return_policy' (input: {\"query\": str}), "
    "'get_refund_status' (input: {\"order_id\": str}), and "
    "'answer' (input: {}). The final step must be 'answer'.")

def make_plan(user_input: str):
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": PLANNER_PROMPT},
            {"role": "user", "content": user_input},
        ],
        response_format={"type": "json_object"},
        temperature=0.2, seed=42)
    return json.loads(resp.choices[0].message.content)["steps"]

def revise_plan(user_input: str, failed_step: dict,
                observation: dict):
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": PLANNER_PROMPT},
            {"role": "user", "content": user_input},
            {"role": "user", "content": (
                f"The step {json.dumps(failed_step)} failed with "
                f"{json.dumps(observation)}. Return a revised plan "
                "for the remaining work.")},
        ],
        response_format={"type": "json_object"},
        temperature=0.2, seed=42)
    return json.loads(resp.choices[0].message.content)["steps"]

def run_plan_and_execute(user_input: str, max_steps: int = 6):
    plan = make_plan(user_input)
    for i, s in enumerate(plan, 1):
        print(f"Plan {i}. {s['action']}({s['input']}) - {s['why']}")
    observations, done = [], 0
    while plan and done < max_steps:
        step = plan.pop(0)
        if step["action"] == "answer":
            break
        obs = TOOL_IMPLEMENTATIONS[step["action"]](**step["input"])
        print(f"Execute: {step['action']} -> {json.dumps(obs)[:60]}")
        observations.append(
            {"action": step["action"], "observation": obs})
        done += 1
        if isinstance(obs, dict) and "error" in obs:
            plan = revise_plan(user_input, step, obs)
            print("Plan revised:",
                  [s["action"] for s in plan])
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": (
                "Answer the user's request using only the "
                "observations. Cite policy passages by their § id.")},
            {"role": "user", "content": user_input},
            {"role": "user", "content": (
                "Observations:\n" + json.dumps(observations))},
        ],
        temperature=0.2, seed=42)
    return resp.choices[0].message.content

**Listing 4.5** — The same request under both regimes: identical grounding, different trace shape and token cost

In [ ]:
REQUEST = ("Has the refund for order A-1001 been issued? And how "
           "long do I have to return another item from the same "
           "delivery?")

_, reactive_tokens = count_tokens(run_agent, REQUEST)
answer, plan_tokens = count_tokens(run_plan_and_execute, REQUEST)
print("\nFinal answer:", answer)
print(f"\nTokens - reactive: {reactive_tokens}, "
      f"plan-and-execute: {plan_tokens}")

## 4.9.3 Reflection: validation re-enters the loop

**Listing 4.6** — A deterministic grounding check: citations in the draft must match the retrieved passages

In [ ]:
def validate_answer(answer: str, retrieved_ids: set):
    cited = set(re.findall(r"§\d+\.\d+", answer))
    if not cited:
        return False, ("The draft cites no policy section. Every "
                       "policy claim must cite a § id from the "
                       "retrieved passages.")
    unsupported = cited - retrieved_ids
    if unsupported:
        return False, (f"The draft cites {sorted(unsupported)}, "
                       "which the run never retrieved.")
    return True, "grounded"

ok, why = validate_answer(
    "You can return items within 14 days.", {"§2.1", "§4.2"})
print(ok, "-", why)

**Listing 4.7** — Reflection: a failed check does not end the run — it re-enters it as feedback

In [ ]:
def run_reflective_agent(user_input: str, max_attempts: int = 3):
    messages = [
        {"role": "system", "content": (
            "You are a customer support agent. Use the tools to "
            "check policies and refund status, then answer.")},
        {"role": "user", "content": user_input},
    ]
    retrieved_ids = set()
    for attempt in range(1, max_attempts + 1):
        for _ in range(6):
            resp = client.chat.completions.create(
                model=CHAT_MODEL, messages=messages,
                tools=[return_policy_tool, refund_status_tool],
                tool_choice="auto", temperature=0.2, seed=42)
            msg = resp.choices[0].message
            if not msg.tool_calls:
                break
            messages.append(
                {"role": "assistant", "content": None,
                     "tool_calls": msg.tool_calls})
            for tc in msg.tool_calls:
                impl = TOOL_IMPLEMENTATIONS[tc.function.name]
                args = json.loads(tc.function.arguments)
                obs = impl(**args)
                if tc.function.name == "search_return_policy":
                    for p in obs["passages"]:
                        retrieved_ids.add(p["id"])
                messages.append(
                    {"role": "tool", "tool_call_id": tc.id,
                     "content": json.dumps(obs)})
        draft = msg.content or ""
        ok, why = validate_answer(draft, retrieved_ids)
        print(f"Attempt {attempt}: validation "
              f"{'passed' if ok else 'failed'} - {why}")
        if ok:
            return draft
        messages.append({"role": "assistant", "content": draft})
        messages.append({"role": "user", "content": (
            f"Your draft failed a grounding check: {why} "
            "Revise the answer accordingly.")})
    return ("Validation kept failing; escalating to a human "
            "with the last draft attached.")

final = run_reflective_agent(
    "How long can I still return an item from my delivery, and "
    "when will I get my money back after returning it?")
print("\nFinal answer:", final)